# M1 Notebook 16 — Sampling Distributions and the Central Limit Theorem

**Notebook ID:** M1_N16  
**Status:** Runnable first edition  
**Random seed:** 42

> A sampling distribution describes how a statistic varies across repeated samples. The Central Limit Theorem explains why sample means often become approximately normal.


## 1. Learning objectives

1. Distinguish a population distribution from a sampling distribution.
2. Simulate sampling distributions of sample means.
3. Compute theoretical standard errors.
4. Demonstrate the Central Limit Theorem for skewed populations.
5. Standardize sample means.
6. Introduce bootstrap distributions and bootstrap standard errors.
7. Connect sampling distributions to inference and Decision Intelligence.


In [ ]:
from srai_math.utils import environment_info, set_seed
from srai_math.probability import (
    bootstrap_standard_error,
    bootstrap_statistic,
    finite_population_correction,
    sample_means,
    sampling_standard_error,
    standardize_sample_means,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
set_seed(42)
environment_info()


## 2. Population, sample, statistic, sampling distribution

A population distribution describes values of a variable in the population.

A sample is a subset drawn from that population.

A statistic, such as

\[
\bar X=\frac{1}{n}\sum_{i=1}^n X_i,
\]

is computed from a sample.

The sampling distribution is the distribution of that statistic over repeated samples.


## 3. Sampling distribution from a normal population

In [ ]:
rng = np.random.default_rng(42)

population_mean = 10.0
population_std = 4.0
sample_size = 25
repetitions = 20_000

means = sample_means(
    lambda shape: rng.normal(
        population_mean,
        population_std,
        size=shape,
    ),
    sample_size=sample_size,
    repetitions=repetitions,
)

summary = {
    "mean_of_sample_means": means.mean(),
    "empirical_standard_error": means.std(ddof=1),
    "theoretical_standard_error": sampling_standard_error(
        population_std,
        sample_size,
    ),
}
summary


In [ ]:
assert np.isclose(means.mean(), population_mean, atol=0.05)
assert np.isclose(
    means.std(ddof=1),
    population_std/np.sqrt(sample_size),
    atol=0.03,
)
print("Sampling mean and standard error verified.")


For independent observations with variance \(\sigma^2\),

\[
\mathbb E[\bar X]=\mu,
\qquad
\operatorname{Var}(\bar X)=\frac{\sigma^2}{n},
\]

so

\[
\operatorname{SE}(\bar X)=\frac{\sigma}{\sqrt n}.
\]


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(means, bins=50, density=True, alpha=0.4)
x = np.linspace(means.min(), means.max(), 500)
ax.plot(
    x,
    stats.norm.pdf(
        x,
        loc=population_mean,
        scale=population_std/np.sqrt(sample_size),
    ),
)
ax.set_xlabel("Sample mean")
ax.set_ylabel("Density")
ax.set_title("Sampling Distribution of the Mean")
plt.show()


## 4. Standard error decreases with sample size

In [ ]:
sample_sizes = np.array([5, 10, 25, 50, 100, 250])
standard_errors = [
    sampling_standard_error(population_std, int(n))
    for n in sample_sizes
]

pd.DataFrame({
    "sample_size": sample_sizes,
    "theoretical_standard_error": standard_errors,
})


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sample_sizes, standard_errors, marker="o")
ax.set_xlabel("Sample size")
ax.set_ylabel("Standard error")
ax.set_title("Standard Error Decreases as 1/sqrt(n)")
plt.show()


## 5. Central Limit Theorem

Under suitable conditions,

\[
rac{ar X-\mu}{\sigma/\sqrt n}
\overset{d}{\longrightarrow}
N(0,1).
\]

This can hold even when the original population is strongly non-normal.


## 6. CLT from an exponential population

In [ ]:
rng = np.random.default_rng(7)
population_rate = 1.0
population_mean_exp = 1.0
population_std_exp = 1.0

clt_results = {}

for n in [1, 5, 30, 100]:
    means_n = sample_means(
        lambda shape, n=n: rng.exponential(
            1/population_rate,
            size=shape,
        ),
        sample_size=n,
        repetitions=20_000,
    )
    clt_results[n] = means_n

fig, ax = plt.subplots(figsize=(8, 5))
for n, values in clt_results.items():
    ax.hist(
        values,
        bins=60,
        density=True,
        histtype="step",
        label=f"n={n}",
    )
ax.set_xlabel("Sample mean")
ax.set_ylabel("Density")
ax.set_title("CLT from a Skewed Exponential Population")
ax.legend()
plt.show()


As \(n\) increases, the sampling distribution becomes more concentrated and more nearly normal.


## 7. Standardized sample means

In [ ]:
means_30 = clt_results[30]
z = standardize_sample_means(
    means_30,
    population_mean=population_mean_exp,
    population_std=population_std_exp,
    sample_size=30,
)

{
    "z_mean": z.mean(),
    "z_standard_deviation": z.std(ddof=1),
}


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(z, bins=50, density=True, alpha=0.4, label="Standardized sample means")
x = np.linspace(-4, 4, 500)
ax.plot(x, stats.norm.pdf(x), label="Standard normal")
ax.set_xlabel("z")
ax.set_ylabel("Density")
ax.set_title("Standardized CLT Approximation")
ax.legend()
plt.show()


## 8. Finite population correction

When sampling without replacement from a finite population,

\[
\operatorname{SE}(ar X)
=
rac{\sigma}{\sqrt n}
\sqrt{rac{N-n}{N-1}}.
\]


In [ ]:
N = 1000
n = 400
fpc = finite_population_correction(N, n)

{
    "finite_population_correction": fpc,
    "uncorrected_SE": sampling_standard_error(20.0, n),
    "corrected_SE": sampling_standard_error(20.0, n) * fpc,
}


## 9. Bootstrap sampling distribution

The bootstrap repeatedly resamples the observed data with replacement to approximate the sampling distribution of a statistic.


In [ ]:
observed_data = np.array([
    12.0, 15.0, 14.0, 18.0, 22.0,
    19.0, 16.0, 17.0, 30.0, 21.0,
])

bootstrap_means = bootstrap_statistic(
    observed_data,
    statistic=np.mean,
    repetitions=10_000,
    seed=42,
)

bootstrap_se = bootstrap_standard_error(
    bootstrap_means,
)

{
    "observed_mean": observed_data.mean(),
    "bootstrap_mean": bootstrap_means.mean(),
    "bootstrap_standard_error": bootstrap_se,
}


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(bootstrap_means, bins=50, density=True)
ax.axvline(observed_data.mean(), linestyle="--")
ax.set_xlabel("Bootstrap sample mean")
ax.set_ylabel("Density")
ax.set_title("Bootstrap Distribution of the Mean")
plt.show()


## 10. Sampling-distribution comparison

In [ ]:
comparison = pd.DataFrame({
    "quantity": [
        "Observed sample standard deviation",
        "Analytical standard error",
        "Bootstrap standard error",
    ],
    "value": [
        observed_data.std(ddof=1),
        observed_data.std(ddof=1)/np.sqrt(observed_data.size),
        bootstrap_se,
    ],
})
comparison


## 11. Statistics interpretation

Sampling distributions underpin:

- confidence intervals;
- standard errors;
- hypothesis tests;
- estimator comparison;
- survey inference;
- uncertainty quantification.


## 12. AI interpretation

Sampling distributions matter in:

- mini-batch variability;
- stochastic gradients;
- cross-validation;
- bootstrap ensembles;
- uncertainty estimation;
- model-performance comparison.


## 13. Decision Intelligence case — Estimating mean service demand

Suppose a public agency samples facilities to estimate average daily service demand.


In [ ]:
rng = np.random.default_rng(123)

facility_population = rng.lognormal(
    mean=3.0,
    sigma=0.55,
    size=5000,
)

sample_n = 100
facility_sample = rng.choice(
    facility_population,
    size=sample_n,
    replace=False,
)

sample_mean = facility_sample.mean()
sample_se = facility_sample.std(ddof=1) / np.sqrt(sample_n)
fpc = finite_population_correction(
    population_size=facility_population.size,
    sample_size=sample_n,
)
corrected_se = sample_se * fpc

{
    "sample_mean_demand": sample_mean,
    "estimated_standard_error": corrected_se,
    "true_population_mean_for_simulation": facility_population.mean(),
}


In [ ]:
bootstrap_demand = bootstrap_statistic(
    facility_sample,
    np.mean,
    repetitions=5000,
    seed=9,
)

quantiles = np.quantile(
    bootstrap_demand,
    [0.025, 0.975],
)

{
    "bootstrap_95_percent_interval": quantiles,
    "bootstrap_standard_error": bootstrap_standard_error(
        bootstrap_demand
    ),
}


### Interpretation

Sampling distributions quantify estimation uncertainty. Operational planning should still account for sample design, nonresponse, clustering, seasonality, data quality, and tail demand.


## 14. Engineering notes

- The CLT is asymptotic, not automatic for every sample size.
- Heavy tails and dependence can slow or invalidate normal approximation.
- Bootstrap validity depends on resampling design and data representativeness.
- Complex survey designs require design-consistent variance estimation.
- Finite population correction matters when the sampling fraction is large.


## 15. Common errors

- Confusing the population distribution with the sampling distribution.
- Using standard deviation and standard error interchangeably.
- Assuming the CLT makes raw data normal.
- Ignoring dependence or clustering.
- Bootstrapping observations that are not exchangeable.
- Treating a narrow standard error as proof of low bias.


## 16. Exercises

### Level A
Explain the difference between a sample distribution and a sampling distribution.

### Level B
Derive the variance of the sample mean.

### Level C
Simulate CLT behavior for Uniform, Poisson, and heavy-tailed populations.

### Capstone
Estimate a public-service mean from a sample, compare analytical and bootstrap uncertainty, apply any relevant finite population correction, and document design limitations.


## 17. Key insight

Sampling distributions describe the uncertainty of statistics. The Central Limit Theorem explains why means often become approximately normal, while bootstrap methods approximate sampling behavior directly from observed data.
